# 12 (heavy) - TotalVI, three times

The authors' integration (4c_TotalVI_CiteSeq+non-CiteSeq.ipynb): 4000 HVG (seurat, batch-aware by sample), TOTALVI n_latent 20,
encoder covariates, lr 1e-3, 400 epochs, train 0.9, batch = sample (scvi-tools 1.5; batch_size 1024, 8 loader workers).
* 03 + 03p: draft (02_qc, 328,041 cells). Record 114 min.
* 08 + 08p: clean counts (07_clean, 321,059). Record 78 min.
* 11 + 11p: strict cells (10_strict, 234,552). Record 112 min.
Post steps (neighbours, UMAP, Leiden) run on GPU with rapids-singlecell. Leiden numbers can differ from the record; the hand
decisions in notebooks 04-06 name record clusters, so compare before re-using them (ARI cell below).

**Heavy notebook.** Nothing runs until you set `RUN_HEAVY = True` in the cell below. Long steps start in a detached
`screen` session with an exit sentinel (`WORK/logs/<name>.exit`); the wait cell polls the log. Re-run the light notebook named
below afterwards so that it uses these rebuilt outputs instead of the record's. Afterwards: notebooks 03 onwards. Each training run needs its input from the previous light notebooks (or the
record): 03 <- 02, 08 <- 04/05 (07_clean), 11 <- 05 (10_strict).

**Runtime:** ~5 h on one T4 GPU
**Envs used (each step runs as a subprocess in its own env):** masld_integration (training), rapids-sc (post)
**Kernel:** `masld_atlas` (orchestration only).

Outputs are written under `/mnt/masld_vault/cross_species_repro/` (mirror of `/mnt/masld_vault/cross_species/`, the outputs
of record). Inputs this notebook does not rebuild are linked from the record; see `repro_setup.py` for the safety rules.

In [ ]:
%run -i /home/ubuntu/liver_cross_species/notebooks/repro_setup.py
print("RECORD", RECORD); print("REPRO ", REPRO); print("WORK  ", WORK)

In [ ]:
RUN_HEAVY = False   # set True to run

In [ ]:
if RUN_HEAVY:   # nothing is touched unless the notebook is really run (review fix)
    guard(30)
    link("guilliams2022/02_qc", "guilliams2022/07_clean", "guilliams2022/10_strict")
    fresh("guilliams2022/03_integration", "guilliams2022/08_totalvi_clean", "guilliams2022/11_totalvi_strict")

### Draft TotalVI

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/03_integrate.py
"""Step 03 - integration of the Fig 1B cells: unintegrated PCA, Harmony baseline, TotalVI (authors' method).

Authors (4c_TotalVI_CiteSeq+non-CiteSeq.ipynb, scvi 0.6 API): union of genes/proteins, 4000 HVG
(seurat_v2, batch-corrected), proteins of non-CITE samples set to 0 and masked per batch, TOTALVI n_latent=20,
encoder_batch=True, lr=1e-3, 400 epochs, train 90 %. Batch = sample.
Here (scvi-tools 1.5): same settings; scvi-tools masks proteins that are all-zero within a batch automatically.

Usage: 03_integrate.py <OUT> [harmony|totalvi|both]
Output: <OUT>/03_integration/
  embeddings.h5ad  obs + obsm X_pca, X_pca_harmony, X_totalvi, X_umap_<method>; no expression matrix
  totalvi_model/   saved model
"""
import os
import sys
import time

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

OUT = sys.argv[1] if len(sys.argv) > 1 else "/mnt/masld_vault/cross_species_repro/guilliams2022"
MODE = sys.argv[2] if len(sys.argv) > 2 else "both"
D = f"{OUT}/03_integration"
os.makedirs(D, exist_ok=True)
t0 = time.time()


def log(m):
    print(f"[{(time.time() - t0) / 60:6.1f} min] {m}", flush=True)


a = ad.read_h5ad(f"{OUT}/02_qc/guilliams2022_fig1b_qc.h5ad")
a.layers["counts"] = a.X.copy()
log(f"loaded {a.shape}")

# HVG: 4000, batch-aware, Seurat flavour on log-normalised data (authors: seurat_v2 with batch correction)
b = a.copy()
sc.pp.normalize_total(b, target_sum=1e4)
sc.pp.log1p(b)
sc.pp.highly_variable_genes(b, n_top_genes=4000, flavor="seurat", batch_key="sample")
hvg = b.var["highly_variable"].values
log(f"HVG {hvg.sum()}")

emb_path = f"{D}/embeddings.h5ad"
E = ad.read_h5ad(emb_path) if os.path.exists(emb_path) else ad.AnnData(obs=a.obs.copy())
assert list(E.obs_names) == list(a.obs_names)

if MODE in ("harmony", "both"):
    c = b[:, hvg].copy()
    sc.pp.scale(c, max_value=10)
    sc.pp.pca(c, n_comps=50, random_state=0)
    E.obsm["X_pca"] = c.obsm["X_pca"]
    sc.pp.neighbors(c, use_rep="X_pca", random_state=0)
    sc.tl.umap(c, random_state=0)
    E.obsm["X_umap_unintegrated"] = c.obsm["X_umap"]
    log("unintegrated PCA/UMAP done")
    import harmonypy
    ho = harmonypy.run_harmony(c.obsm["X_pca"], c.obs, ["sample"], random_state=0)
    Z = np.asarray(ho.Z_corr)
    Z = Z.T if Z.shape[0] != c.n_obs else Z
    c.obsm["X_pca_harmony"] = Z
    E.obsm["X_pca_harmony"] = Z
    sc.pp.neighbors(c, use_rep="X_pca_harmony", random_state=0)
    sc.tl.umap(c, random_state=0)
    E.obsm["X_umap_harmony"] = c.obsm["X_umap"]
    log("harmony done")
    E.write_h5ad(emb_path)
    del c

if MODE in ("totalvi", "both"):
    import scvi
    import torch
    scvi.settings.seed = 0
    t = a[:, hvg].copy()
    t.obsm["protein_expression"] = a.obsm["adt"].copy()
    scvi.model.TOTALVI.setup_anndata(t, layer="counts", batch_key="sample",
                                     protein_expression_obsm_key="protein_expression")
    m = scvi.model.TOTALVI(t, n_latent=20, encode_covariates=True)
    log(f"TOTALVI set up; cuda={torch.cuda.is_available()}")
    from lightning.pytorch.callbacks import Callback

    class EpochLog(Callback):  # one line per 5 epochs with s/epoch and ETA
        def __init__(self):
            self.t = time.time()

        def on_train_epoch_end(self, trainer, pl_module):
            e = trainer.current_epoch + 1
            if e % 5 == 0 or e == 1:
                dt = time.time() - self.t
                print(f"epoch {e}/{trainer.max_epochs} {dt / e:.1f} s/epoch, "
                      f"eta {dt / e * (trainer.max_epochs - e) / 60:.0f} min", flush=True)

    # 2026-11-11 restart (user option 2): batch 1024 + 8 loader workers instead of scvi default 256 / 0 workers;
    # the 256 run was loader-bound (GPU ~38 %). Deviation from the authors' training settings, draft only.
    scvi.settings.dl_num_workers = 8
    m.train(max_epochs=400, lr=1e-3, batch_size=1024, train_size=0.9, accelerator="gpu", devices=1,
            enable_progress_bar=False, early_stopping=False, callbacks=[EpochLog()])
    log("TOTALVI trained")
    m.save(f"{D}/totalvi_model", overwrite=True)
    hist = pd.concat([v for v in m.history.values() if isinstance(v, pd.DataFrame)], axis=1)
    hist.to_csv(f"{D}/totalvi_history.tsv", sep="\t")
    E.obsm["X_totalvi"] = m.get_latent_representation()
    E.write_h5ad(emb_path)  # UMAP + Leiden on GPU in 03p_post_gpu.py
    log("latent written")

log("finished")

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/03p_post_gpu.py
"""Step 03p - neighbours, UMAP and Leiden (r=1) on the TotalVI latent, on GPU (rapids-singlecell).
Updates <OUT>/03_integration/embeddings.h5ad with X_umap_totalvi and leiden_totalvi_r1."""
import sys

import anndata as ad
import numpy as np
import rapids_singlecell as rsc

OUT = sys.argv[1] if len(sys.argv) > 1 else "/mnt/masld_vault/cross_species_repro/guilliams2022"
p = f"{OUT}/03_integration/embeddings.h5ad"
E = ad.read_h5ad(p)
t = ad.AnnData(obs=E.obs[[]].copy(), obsm={"X_totalvi": E.obsm["X_totalvi"].astype(np.float32)})
rsc.pp.neighbors(t, use_rep="X_totalvi")
rsc.tl.umap(t, random_state=0)
rsc.tl.leiden(t, resolution=1.0, random_state=0)
E.obsm["X_umap_totalvi"] = np.asarray(t.obsm["X_umap"])
E.obs["leiden_totalvi_r1"] = t.obs["leiden"].astype(str).values
E.write_h5ad(p)
print("written", E.shape, E.obs["leiden_totalvi_r1"].nunique(), "clusters", flush=True)

### TotalVI on the clean counts

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/08_totalvi_clean.py
"""Step 08 - TotalVI on the cleaned counts (07_clean, 321,059 cells/nuclei), same settings as the draft (03_integrate.py):
4000 HVG (seurat flavour, batch-aware by sample) on log-normalised cleaned counts, TOTALVI n_latent 20, encode_covariates,
lr 1e-3, 400 epochs, train 0.9, batch = sample, batch_size 1024 + 8 loader workers (draft deviation kept for comparability).
Counts = X of the clean file (decontX (1,300) for 13 in vivo samples, raw otherwise; integers). Proteins = obsm adt
(all-zero in non-CITE batches -> masked by scvi-tools).
Then on GPU: neighbours, UMAP, Leiden r1/r2/r3; UMAP panel of the curated draft labels.
Output <OUT>/08_totalvi_clean/: embeddings.h5ad (obs + X_totalvi, X_umap_totalvi, leiden), totalvi_model/, history, figures/."""
import os
import time

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

OUT = "/mnt/masld_vault/cross_species_repro/guilliams2022"
D = f"{OUT}/08_totalvi_clean"
os.makedirs(f"{D}/figures", exist_ok=True)
t0 = time.time()


def log(m):
    print(f"[{(time.time() - t0) / 60:6.1f} min] {m}", flush=True)


a = ad.read_h5ad(f"{OUT}/07_clean/guilliams2022_fig1b_clean.h5ad")
a.layers.pop("counts")  # raw; not used here
a.layers["clean_counts"] = a.X.copy()
log(f"loaded {a.shape}")
b = a.copy()
sc.pp.normalize_total(b, target_sum=1e4)
sc.pp.log1p(b)
sc.pp.highly_variable_genes(b, n_top_genes=4000, flavor="seurat", batch_key="sample")
hvg = b.var["highly_variable"].values
del b
log(f"HVG {hvg.sum()}")
pd.Series(a.var_names[hvg]).to_csv(f"{D}/hvg.tsv", sep="\t", index=False)

import scvi  # noqa: E402
import torch  # noqa: E402
from lightning.pytorch.callbacks import Callback  # noqa: E402

scvi.settings.seed = 0
t = a[:, hvg].copy()
t.obsm["protein_expression"] = np.asarray(a.obsm["adt"])
scvi.model.TOTALVI.setup_anndata(t, layer="clean_counts", batch_key="sample", protein_expression_obsm_key="protein_expression")
m = scvi.model.TOTALVI(t, n_latent=20, encode_covariates=True)
log(f"TOTALVI set up; cuda={torch.cuda.is_available()}")


class EpochLog(Callback):
    def __init__(self):
        self.t = time.time()

    def on_train_epoch_end(self, trainer, pl_module):
        e = trainer.current_epoch + 1
        if e % 5 == 0 or e == 1:
            dt = time.time() - self.t
            print(f"epoch {e}/{trainer.max_epochs} {dt / e:.1f} s/epoch, eta {dt / e * (trainer.max_epochs - e) / 60:.0f} min", flush=True)


scvi.settings.dl_num_workers = 8
m.train(max_epochs=400, lr=1e-3, batch_size=1024, train_size=0.9, accelerator="gpu", devices=1,
        enable_progress_bar=False, early_stopping=False, callbacks=[EpochLog()])
log("TOTALVI trained")
m.save(f"{D}/totalvi_model", overwrite=True)
pd.concat([v for v in m.history.values() if isinstance(v, pd.DataFrame)], axis=1).to_csv(f"{D}/totalvi_history.tsv", sep="\t")
Z = m.get_latent_representation()
del m, t
torch.cuda.empty_cache()
E = ad.AnnData(obs=a.obs.copy(), obsm={"X_totalvi": Z.astype(np.float32)})
for k in E.obs.columns:
    if E.obs[k].dtype == object:
        E.obs[k] = E.obs[k].astype(str)
E.write_h5ad(f"{D}/embeddings.h5ad")
log("latent written; UMAP/Leiden/panel in 08p_post_gpu.py")

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/08p_post_gpu.py
"""Step 08p - GPU neighbours/UMAP/Leiden r1-r3 on the clean TotalVI latent + UMAP panel (rapids-sc env)."""
import anndata as ad
import numpy as np
import scanpy as sc
import rapids_singlecell as rsc

OUT = "/mnt/masld_vault/cross_species_repro/guilliams2022"
D = f"{OUT}/08_totalvi_clean"
E = ad.read_h5ad(f"{D}/embeddings.h5ad")
a = ad.read_h5ad(f"{OUT}/07_clean/guilliams2022_fig1b_clean.h5ad")
assert (E.obs_names == a.obs_names).all()
a.layers.pop("counts")
def log(m):
    print(m, flush=True)

g = ad.AnnData(obs=E.obs[[]].copy(), obsm={"X_totalvi": E.obsm["X_totalvi"]})
rsc.pp.neighbors(g, use_rep="X_totalvi")
rsc.tl.umap(g, random_state=0)
E.obsm["X_umap_totalvi"] = np.asarray(g.obsm["X_umap"])
for r in (1.0, 2.0, 3.0):
    rsc.tl.leiden(g, resolution=r, random_state=0, key_added="l")
    E.obs[f"leiden_clean_r{r:g}"] = g.obs["l"].astype(str).values
for k in E.obs.columns:
    if E.obs[k].dtype == object:
        E.obs[k] = E.obs[k].astype(str)
E.write_h5ad(f"{D}/embeddings.h5ad")
log(f"embeddings written; clusters r1/r2/r3 {[E.obs[f'leiden_clean_r{r}'].nunique() for r in (1, 2, 3)]}")

import matplotlib  # noqa: E402

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

n = a.copy()
sc.pp.normalize_total(n, target_sum=1e4)
sc.pp.log1p(n)
n.obsm["X_umap"] = E.obsm["X_umap_totalvi"]
n.obs = E.obs
n.var_names = n.var["symbol"].astype(str).values
n.var_names_make_unique()
cols = ["annotation_curated", "annotation_flag", "suspension", "digestion", "decontx_setting", "leiden_clean_r1",
        "Alb", "Clec4f", "Ptprc", "Stab2", "Reln", "Krt19", "Cd3e", "Cd79a", "S100a8", "Siglech"]
fig, axs = plt.subplots(4, 4, figsize=(28, 26))
for ax, c in zip(axs.ravel(), cols):
    sc.pl.umap(n, color=c, ax=ax, show=False, frameon=False, s=1,
               legend_loc="on data" if c in ("annotation_curated", "leiden_clean_r1") else "right margin", legend_fontsize=6)
    for coll in ax.collections:
        coll.set_rasterized(True)
fig.suptitle("TotalVI on cleaned counts (07_clean, 321,059) - curated draft labels", fontsize=14)
fig.savefig(f"{D}/figures/umap_totalvi_clean_panel.png", dpi=150, bbox_inches="tight")
fig.savefig(f"{D}/figures/umap_totalvi_clean_panel.pdf", dpi=300, bbox_inches="tight")
log("finished")

### TotalVI on the strict cells

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/11_totalvi_strict.py
"""Step 11 - TotalVI on the strict-QC cells (10_strict), same settings as the draft (03_integrate.py):
4000 HVG (seurat flavour, batch-aware by sample) on log-normalised cleaned counts, TOTALVI n_latent 20, encode_covariates,
lr 1e-3, 400 epochs, train 0.9, batch = sample, batch_size 1024 + 8 loader workers (draft deviation kept for comparability).
Counts = X of the clean file (decontX (1,300) for 13 in vivo samples, raw otherwise; integers). Proteins = obsm adt
(all-zero in non-CITE batches -> masked by scvi-tools).
Then on GPU: neighbours, UMAP, Leiden r1/r2/r3; UMAP panel of the curated draft labels.
Output <OUT>/11_totalvi_strict/: embeddings.h5ad (obs + X_totalvi, X_umap_totalvi, leiden), totalvi_model/, history, figures/."""
import os
import time

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

OUT = "/mnt/masld_vault/cross_species_repro/guilliams2022"
D = f"{OUT}/11_totalvi_strict"
os.makedirs(f"{D}/figures", exist_ok=True)
t0 = time.time()


def log(m):
    print(f"[{(time.time() - t0) / 60:6.1f} min] {m}", flush=True)


a = ad.read_h5ad(f"{OUT}/10_strict/guilliams2022_fig1b_strict.h5ad")
a.layers.pop("counts")  # raw; not used here
a.layers["clean_counts"] = a.X.copy()
log(f"loaded {a.shape}")
b = a.copy()
sc.pp.normalize_total(b, target_sum=1e4)
sc.pp.log1p(b)
sc.pp.highly_variable_genes(b, n_top_genes=4000, flavor="seurat", batch_key="sample")
hvg = b.var["highly_variable"].values
del b
log(f"HVG {hvg.sum()}")
pd.Series(a.var_names[hvg]).to_csv(f"{D}/hvg.tsv", sep="\t", index=False)

import scvi  # noqa: E402
import torch  # noqa: E402
from lightning.pytorch.callbacks import Callback  # noqa: E402

scvi.settings.seed = 0
t = a[:, hvg].copy()
t.obsm["protein_expression"] = np.asarray(a.obsm["adt"])
scvi.model.TOTALVI.setup_anndata(t, layer="clean_counts", batch_key="sample", protein_expression_obsm_key="protein_expression")
m = scvi.model.TOTALVI(t, n_latent=20, encode_covariates=True)
log(f"TOTALVI set up; cuda={torch.cuda.is_available()}")


class EpochLog(Callback):
    def __init__(self):
        self.t = time.time()

    def on_train_epoch_end(self, trainer, pl_module):
        e = trainer.current_epoch + 1
        if e % 5 == 0 or e == 1:
            dt = time.time() - self.t
            print(f"epoch {e}/{trainer.max_epochs} {dt / e:.1f} s/epoch, eta {dt / e * (trainer.max_epochs - e) / 60:.0f} min", flush=True)


scvi.settings.dl_num_workers = 8
m.train(max_epochs=400, lr=1e-3, batch_size=1024, train_size=0.9, accelerator="gpu", devices=1,
        enable_progress_bar=False, early_stopping=False, callbacks=[EpochLog()])
log("TOTALVI trained")
m.save(f"{D}/totalvi_model", overwrite=True)
pd.concat([v for v in m.history.values() if isinstance(v, pd.DataFrame)], axis=1).to_csv(f"{D}/totalvi_history.tsv", sep="\t")
Z = m.get_latent_representation()
del m, t
torch.cuda.empty_cache()
E = ad.AnnData(obs=a.obs.copy(), obsm={"X_totalvi": Z.astype(np.float32)})
for k in E.obs.columns:
    if E.obs[k].dtype == object:
        E.obs[k] = E.obs[k].astype(str)
E.write_h5ad(f"{D}/embeddings.h5ad")
log("latent written; UMAP/Leiden/panel in 08p_post_gpu.py")

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/11p_post_gpu.py
"""Step 11p - GPU neighbours/UMAP/Leiden r1-r3 on the clean TotalVI latent + UMAP panel (rapids-sc env)."""
import anndata as ad
import numpy as np
import scanpy as sc
import rapids_singlecell as rsc

OUT = "/mnt/masld_vault/cross_species_repro/guilliams2022"
D = f"{OUT}/11_totalvi_strict"
E = ad.read_h5ad(f"{D}/embeddings.h5ad")
a = ad.read_h5ad(f"{OUT}/10_strict/guilliams2022_fig1b_strict.h5ad")
assert (E.obs_names == a.obs_names).all()
a.layers.pop("counts")
def log(m):
    print(m, flush=True)

g = ad.AnnData(obs=E.obs[[]].copy(), obsm={"X_totalvi": E.obsm["X_totalvi"]})
rsc.pp.neighbors(g, use_rep="X_totalvi")
rsc.tl.umap(g, random_state=0)
E.obsm["X_umap_totalvi"] = np.asarray(g.obsm["X_umap"])
for r in (1.0, 2.0, 3.0):
    rsc.tl.leiden(g, resolution=r, random_state=0, key_added="l")
    E.obs[f"leiden_clean_r{r:g}"] = g.obs["l"].astype(str).values
for k in E.obs.columns:
    if E.obs[k].dtype == object:
        E.obs[k] = E.obs[k].astype(str)
E.write_h5ad(f"{D}/embeddings.h5ad")
log(f"embeddings written; clusters r1/r2/r3 {[E.obs[f'leiden_clean_r{r}'].nunique() for r in (1, 2, 3)]}")

import matplotlib  # noqa: E402

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

n = a.copy()
sc.pp.normalize_total(n, target_sum=1e4)
sc.pp.log1p(n)
n.obsm["X_umap"] = E.obsm["X_umap_totalvi"]
n.obs = E.obs
n.var_names = n.var["symbol"].astype(str).values
n.var_names_make_unique()
cols = ["annotation_curated", "annotation_flag", "suspension", "digestion", "decontx_setting", "leiden_clean_r1",
        "Alb", "Clec4f", "Ptprc", "Stab2", "Reln", "Krt19", "Cd3e", "Cd79a", "S100a8", "Siglech"]
fig, axs = plt.subplots(4, 4, figsize=(28, 26))
for ax, c in zip(axs.ravel(), cols):
    sc.pl.umap(n, color=c, ax=ax, show=False, frameon=False, s=1,
               legend_loc="on data" if c in ("annotation_curated", "leiden_clean_r1") else "right margin", legend_fontsize=6)
    for coll in ax.collections:
        coll.set_rasterized(True)
fig.suptitle("TotalVI on strict-QC cells (10_strict) - curated draft labels", fontsize=14)
fig.savefig(f"{D}/figures/umap_totalvi_clean_panel.png", dpi=150, bbox_inches="tight")
fig.savefig(f"{D}/figures/umap_totalvi_clean_panel.pdf", dpi=300, bbox_inches="tight")
log("finished")

In [ ]:
MI, RS, O = PY["masld_integration"], PY["rapids-sc"], rep("guilliams2022")
launch("totalvi", ["export CUDA_VISIBLE_DEVICES=0", "set -e",
    f"{MI} -I {WORK}/03_integrate.py {O} totalvi > {LOGS}/03_integrate.py.log 2>&1",
    f"{RS} -I {WORK}/03p_post_gpu.py {O} > {LOGS}/03p_post_gpu.py.log 2>&1",
    f"{MI} -I {WORK}/08_totalvi_clean.py > {LOGS}/08_totalvi_clean.py.log 2>&1",
    f"{RS} -I {WORK}/08p_post_gpu.py > {LOGS}/08p_post_gpu.py.log 2>&1",
    f"{MI} -I {WORK}/11_totalvi_strict.py > {LOGS}/11_totalvi_strict.py.log 2>&1",
    f"{RS} -I {WORK}/11p_post_gpu.py > {LOGS}/11p_post_gpu.py.log 2>&1"])
wait_for("totalvi", every=600)

In [ ]:
if RUN_HEAVY:
    import anndata as ad
    from sklearn.metrics import adjusted_rand_score as ari
    for f, k in [("03_integration/embeddings.h5ad", "leiden_totalvi_r1"), ("08_totalvi_clean/embeddings.h5ad", "leiden_clean_r3"),
                 ("11_totalvi_strict/embeddings.h5ad", "leiden_clean_r3")]:
        a = ad.read_h5ad(rec(f"guilliams2022/{f}"), backed="r").obs[k]; b = ad.read_h5ad(rep(f"guilliams2022/{f}"), backed="r").obs[k]
        print(f, k, "ARI vs record", round(ari(a.astype(str), b.reindex(a.index).astype(str)), 3))